In [ ]:
# Device Setup

import torch 
from transformers import AutoTokenizer , AutoModelForSequenceClassification

device = torch.device('mps' if torch.mps.is_available() else 'cpu')
print('🚀 Using device : ' , device)

🚀 Using device :  mps


In [4]:
# Load Teacher Model : Use Pre-trained Model that be trained on SST-2 dataset (emotional classify dataset)
model_name = 'textattack/bert-base-uncased-SST-2'

print('🕐 Loading Teacher Model ...')
tokenizer = AutoTokenizer.from_pretrained(model_name) # โหลดตัวแปลงข้อความของโมเดลข้างบน เพื่อแปลง
model = AutoModelForSequenceClassification.from_pretrained(model_name) # โหลดโมเดล สำหรับจัดข้อความ

model.to(device)

model.eval() # change model to Inference mode : โหมดใช้งาน --> ใช้ตอนไม่เทรน (ปืด dropout , ไม่ปรับ parameter)
print('✅ Teacher Model is Ready !')


🕐 Loading Teacher Model ...
✅ Teacher Model is Ready !


In [8]:
# Prepare Data Test 
text_inputs = ["I love this movie ! Its awesome" , "This movie is really stupid and bored"]

# change text into tensor (number)
inputs = tokenizer(
    text_inputs , 
    padding = True , # add 0 to short sentence = long sentence
    truncation = True , # if sentences too long its will cut it (truncation : การตัดทอน) 
    # เช็คจาก max_length เช่น tokenization = 512 max_length = 300 ตัด 212 ออก

    return_tensors = "pt" # request output to pytorch tensor if dont have this will return only list
    # ถ้าได้ torch.Tensor มาก็พร้อม input เข้า โมเดลได้ทันที
)

# move input to same devices (loop all value to device)
inputs = {k: v.to(device) for k , v in inputs.items()} 
# ปกติใน inputs จาก tokenizer จะมี 2 อย่าง เช่น { "input_ids": tensor, "attention_mask": tensor }
print(f'Example Inputs : {inputs.items()}')

# tips : everything that have parameter and calculate will sent to gpu / mps
# 1 . model 
# 2 . Tensor

Example Inputs : dict_items([('input_ids', tensor([[  101,  1045,  2293,  2023,  3185,   999,  2049, 12476,   102],
        [  101,  2023,  3185,  2003,  2428,  5236,  1998, 11471,   102]],
       device='mps:0')), ('token_type_ids', tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]], device='mps:0')), ('attention_mask', tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0'))])


In [ ]:
# Running Inferences

# run model : no need to calculate gradient
with torch.no_grad() :
    output = model(**inputs) # --> sent all in dict to model function

logits = output.logits # raw score we need

print(f'--- 📊 Results ----')
print(f'Logit Shape ->> {logits.shape}') # its might be [2,2] # 2 sentences , 2 classes

print('-----')
print(f'Raw logit : {logits}')
print('-----')

#change to probability : use softmax
probs = torch.softmax(logits , dim = 1)
print(f'Probability : {probs}')
print('-----')

# summarize 
labels = ['Negative' , 'Positive'] # classes

for i , sent in enumerate(text_inputs) :
    predicted_id = torch.argmax(logits[i]).item() # find max score position
    print(f'Sentence : {sent}')
    print(f'predict : {labels[predicted_id]} , Score : {logits[i][predicted_id]:.4f}')


--- 📊 Results ----
Logit Shape ->> torch.Size([2, 2])
-----
Raw logit : tensor([[-4.0835,  3.8783],
        [ 3.8706, -3.5090]], device='mps:0')
-----
Probability : tensor([[3.4841e-04, 9.9965e-01],
        [9.9938e-01, 6.2350e-04]], device='mps:0')
-----
Sentence : I love this movie ! Its awesome
predict : Positive , Score : 3.8783
Sentence : This movie is really stupid and bored
predict : Negative , Score : 3.8706


How to Read Tensors
Row : 1
[-4.0835,  3.8783]

class 0 = -4.08
class 1 = 3.88 ← higher

👉 Model choose -> class 1

Row : 2
[ 3.8706, -3.5090]

class 0 = 3.87 ← higher
class 1 = -3.51

👉 Model choose -> class 0